Creating my weighted loss entropy

In [4]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Load the dataset (matching the logic from your EDA)
cols = ['id', 'article_id', 'keyword', 'country', 'text', 'label']
df = pd.read_csv('dontpatronizeme_pcl.tsv', sep='\t', names=cols, skiprows=4)

# 2. Clean and define the binary task
df = df.dropna(subset=['text', 'label'])
df['label'] = df['label'].astype(int)
df['label'] = df['label'].apply(lambda x: 1 if x >= 2 else 0)

# 3. Create Train and Dev splits
# Note: The spec mentions an official allocation file[cite: 120].
# If you downloaded that, use it to split your dataframe.
# Otherwise, here is a standard 80/20 split using stratification to maintain class balance:
train_df, dev_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 4. Convert Pandas DataFrames to Hugging Face Datasets
train_dataset_hf = Dataset.from_pandas(train_df)
dev_dataset_hf = Dataset.from_pandas(dev_df)

# 5. Define the tokenization function
# This converts your raw text into the numbers (input_ids) RoBERTa expects
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128 # You can adjust this based on the max sentence length you found in EDA
    )

# 6. Apply tokenization to both datasets
train_dataset = train_dataset_hf.map(tokenize_function, batched=True)
dev_dataset = dev_dataset_hf.map(tokenize_function, batched=True)

# 7. Format the datasets for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
dev_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("Datasets are ready!")

Map:   0%|          | 0/8374 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Datasets are ready!


In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import f1_score

class WeightedTrainer(Trainer):
    # Added num_items_in_batch and **kwargs to handle the new library version
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculate weights based on your EDA
        weights = torch.tensor([1.0, 8.5]).to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return {"f1": f1_score(labels, predictions, pos_label=1)}

training_args = TrainingArguments(
    output_dir="./BestModel",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss


In [ ]:
import numpy as np

# 1. Generate predictions for the Dev Set
dev_predictions_output = trainer.predict(tokenized_dev)
# Extract logits and get the highest probability class (0 or 1)
dev_preds = np.argmax(dev_predictions_output.predictions, axis=-1)

# 2. Generate predictions for the Test Set
test_predictions_output = trainer.predict(tokenized_test)
test_preds = np.argmax(test_predictions_output.predictions, axis=-1)

# 3. Save to dev.txt
with open("dev.txt", "w") as f:
    for pred in dev_preds:
        f.write(f"{pred}\n")
print(f"Saved {len(dev_preds)} predictions to dev.txt")

# 4. Save to test.txt
# The coursework spec mentions the test set has exactly 3832 lines.
# This is a great sanity check!
with open("test.txt", "w") as f:
    for pred in test_preds:
        f.write(f"{pred}\n")
print(f"Saved {len(test_preds)} predictions to test.txt")

# Quick Sanity Check
if len(test_preds) == 3832:
    print("Success! Test predictions match the expected 3832 lines.")
else:
    print("Warning: The number of test predictions does not match 3832.")